# Codificação

A implementação baseia-se na ideia de utilizar a **Decomposição em Valores Singulares (SVD)** da imagem para incorporar uma mensagem secreta diretamente em seus valores singulares.


# Importações

In [ ]:
from matplotlib.image import imread
import matplotlib.pyplot as plt
import numpy as np
import os

# Funções auxiliares

Inicialmente, o usuário fornece a mensagem secreta por meio da função `input()`. Essa mensagem é convertida para sua representação binária através da função `texto_para_bits()`, resultando em uma sequência de bits que será efetivamente incorporada à imagem. Trabalhar com bits simplifica o processo de codificação, pois cada bit pode ser associado a uma pequena modificação em um valor singular.

In [ ]:
def print_matrix(M, name="Matriz", decimals=6):
    with np.printoptions(
        precision=decimals,
        suppress=True,
        linewidth=200,
        threshold=np.inf
    ):
        print(f"\n{name} ({M.shape[0]} x {M.shape[1]})")
        print(M)


def texto_para_bits(texto):
    texto += '\0'   # caractere de fim
    return ''.join(format(ord(c), '08b') for c in texto)


# Definida aqui também (além da seção de Decodificação) porque
# é usada pelo teste exploratório de robustez a JPEG, que roda
# ainda dentro da etapa de codificação.
def bits_para_texto(bits):
    mensagem = ""
    for i in range(0, len(bits), 8):
        byte = bits[i:i+8]
        if len(byte) < 8:
            break
        caractere = chr(int(byte, 2))
        if caractere == '\0':
            break
        mensagem += caractere
    return mensagem

# Pré-Processamento

Em seguida, a imagem é carregada e convertida para tons de cinza. Essa conversão é realizada calculando a média dos três canais de cor (RGB), produzindo uma matriz bidimensional $X$, na qual cada elemento representa a intensidade de um pixel. Essa matriz é a entrada para a decomposição SVD.

In [ ]:
# Leitura da mensagem
mensagem = input("Digite a mensagem secreta: ")
bits = texto_para_bits(mensagem)

print("\nMensagem:")
print(mensagem)
print("\nBits:")
print(bits)

# Leitura da imagem
plt.rcParams['figure.figsize'] = [15,8]

A = imread(os.path.join("DATA", "Passaro.jpg"))

# converte para tons de cinza
X = np.mean(A, axis=2)
X = np.mean(A, axis=2)
print(f"Dimensões da imagem: {X.shape[0]} x {X.shape[1]} pixels")
plt.imshow(X, cmap='gray')
plt.title("Imagem Original")
plt.axis('off')
plt.show()

In [ ]:
# Mantemos uma cópia dos bits originais apenas para fins de avaliação (taxa de erro de bits) mais adiante. Em um cenário real de esteganografia cega, o receptor NÃO tem acesso a esta variável.
bits_originais = bits

# Decomposição SVD

In [ ]:
U, s, VT = np.linalg.svd(X, full_matrices=False)
s_estego = s.copy()

In [ ]:
import time

inicio_tempo = time.time()
U, s, VT = np.linalg.svd(X, full_matrices=False)
tempo_svd = time.time() - inicio_tempo

s_estego = s.copy()
print(f"Tempo de execução da SVD: {tempo_svd:.4f} s  (imagem {X.shape[0]}x{X.shape[1]})")

# Número de Condição

O número de condição na norma 2, $\kappa_2(X) = \sigma_{\max}(X) / \sigma_{\min}(X)$, mede a sensibilidade da matriz da imagem a perturbações (Golub e Van Loan, Seção 2.6). Valores elevados indicam maior risco de que perturbações pequenas nos valores singulares alterem significativamente a estrutura recuperada.

In [ ]:
cond = np.linalg.cond(X)
print(f"Número de condição κ₂(X) = {cond:.2e}")

# Verificação da capacidade

Antes da codificação, é realizada uma verificação de capacidade. Como cada bit da mensagem é armazenado em um valor singular, o número de bits da mensagem não pode exceder a quantidade de valores singulares disponíveis. Caso isso ocorra, o algoritmo interrompe a execução, evitando perda de informação.

In [ ]:
if len(bits) > len(s_estego):
    raise ValueError("Mensagem muito grande para esta imagem.")

capacidade_teorica = len(s_estego)
capacidade_usada = len(bits)
print(f"Bits inseridos: {capacidade_usada} / {capacidade_teorica} "
      f"({100*capacidade_usada/capacidade_teorica:.2f}% da capacidade)")

# Codificação da Imagem

A decomposição em valores singulares é então aplicada à matriz da imagem, produzindo

$$
X = U \Sigma V^T,
$$

em que $U$ e $V^T$ são matrizes ortogonais e $\Sigma$ é uma matriz diagonal contendo os valores singulares da imagem. Em vez de modificar diretamente os pixels, a implementação altera apenas elementos da diagonal de $\Sigma$, preservando as direções principais representadas pelas matrizes $U$ e $V^T$.

**Proteção dos maiores valores singulares.** Como discutido na Fundamentação Teórica, os maiores valores singulares concentram a maior parte da informação estrutural da imagem, enquanto os menores contribuem apenas com detalhes finos e imperceptíveis (Bergman e Davidson, 2005). Por isso, a mensagem de $L$ bits é inserida apenas nos $L$ **menores** valores singulares (índices $n-L, \ldots, n-1$, considerando $\Sigma$ ordenada de forma decrescente), deixando os $n-L$ maiores completamente intactos.

A etapa de codificação utiliza uma estratégia baseada na **paridade** dos valores singulares. Inicialmente, cada valor singular selecionado é multiplicado por um fator de escala (igual a 100), convertendo-o em um número com maior precisão inteira. Em seguida, para cada bit da mensagem:

- Se o bit for **0**, o algoritmo garante que o valor inteiro correspondente seja **par**;
- Se o bit for **1**, o algoritmo garante que o valor inteiro correspondente seja **ímpar**.

Quando necessário, apenas uma unidade é adicionada ou subtraída do valor inteiro para alterar sua paridade. Após esse ajuste, o valor é dividido novamente pelo fator de escala, retornando aproximadamente à sua magnitude original.

**Verificação de ordenação (robustez).** Como os valores singulares perturbados precisam permanecer em ordem não-crescente (exigência da própria definição de SVD), o algoritmo verifica, após a inserção, se a perturbação de ±0,01 não inverteu a ordem de dois valores singulares vizinhos muito próximos entre si. Se dois valores estiverem a uma distância menor que a perturbação máxima possível, a ordem pode ser invertida, corrompendo silenciosamente a correspondência entre índice e bit na extração — um risco real na cauda do espectro de valores singulares, onde eles tendem a ficar mais próximos uns dos outros (fenômeno relacionado à "instabilidade de autovetores próximos" descrita por Golub e Van Loan, Seção 2.6).

In [ ]:
escala = 100
n = len(s_estego)
L = len(bits)
inicio = n - L  # menor índice utilizado para a mensagem
assert L <= n, "Mensagem excede a capacidade da imagem (verifique a célula de capacidade acima)."

# Inserção a partir do MENOR valor singular (índice n-1) em direção
# aos maiores, até o índice n-L. Isso mantém o método cego: a
# extração (mais adiante) percorre os valores singulares na mesma
# direção, sem precisar conhecer L previamente.
for k, bit in enumerate(bits):
    i = n - 1 - k
    valor = int(round(s_estego[i] * escala))
    if bit == '0' and valor % 2 == 1:
        valor -= 1
    elif bit == '1' and valor % 2 == 0:
        valor += 1
    s_estego[i] = valor / escala

# Verificação de robustez: a perturbação não pode inverter a ordem
# decrescente dos valores singulares na região de inserção, pois
# isso corromperia a correspondência índice <-> bit na extração.
regiao = s_estego[max(inicio - 1, 0):]
if np.any(np.diff(regiao) > 0):
    raise RuntimeError(
        "A perturbação alterou a ordem dos valores singulares na "
        "região de inserção. Reduza 'escala', reduza o tamanho da "
        "mensagem, ou utilize a técnica de espaçamento de valores "
        "singulares (Bergman e Davidson, Seção 5.1) antes de "
        "inserir os bits."
    )

In [ ]:
regiao_usada = s[n - len(bits):]
gaps = np.abs(np.diff(regiao_usada))
print("Menor distância entre valores singulares vizinhos na região usada:", gaps.min())
print("Posição do menor gap (índice relativo):", gaps.argmin())
print("Valores envolvidos:", regiao_usada[gaps.argmin()], regiao_usada[gaps.argmin()+1])

In [ ]:
count_perigosos = np.sum(gaps < 0.02)
print(f"Pares de valores singulares mais próximos que o limite seguro: {count_perigosos} de {len(gaps)}")

# Reconstrução da imagem

Após modificar os valores singulares, uma nova matriz diagonal $\Sigma_{\text{estego}}$ é construída e a imagem esteganográfica é reconstruída utilizando

$$
X_{\text{estego}} = U \Sigma_{\text{estego}} V^T.
$$

Como apenas pequenas variações foram introduzidas nos valores singulares, a imagem reconstruída mantém aparência muito semelhante à imagem original.

Para quantificar a diferença entre as duas imagens, o algoritmo calcula o **erro de Frobenius**

$$
\|X - X_{\text{estego}}\|_F,
$$

que mede a diferença global entre as matrizes da imagem original e da imagem modificada. Valores pequenos dessa norma indicam que as alterações introduzidas pela esteganografia são mínimas.

In [ ]:
S_estego = np.diag(s_estego)
X_estego = U @ S_estego @ VT

# Erro de Frobenius

In [ ]:
erro = np.linalg.norm(X - X_estego)
print(f"\nErro de Frobenius = {erro:.6f}")

# PSNR e SSIM

Além do erro de Frobenius, calculamos duas métricas padrão de qualidade de imagem para avaliar objetivamente a imperceptibilidade da esteganografia: o PSNR (*Peak Signal-to-Noise Ratio*) e o SSIM (*Structural Similarity Index*).

In [ ]:
def calcular_psnr(original, comparada, max_valor=255):
    mse = np.mean((original.astype(np.float64) - comparada.astype(np.float64)) ** 2)
    if mse == 0:
        return float('inf')
    return 10 * np.log10((max_valor ** 2) / mse)

psnr = calcular_psnr(X, X_estego)
print(f"PSNR = {psnr:.2f} dB")

try:
    from skimage.metrics import structural_similarity
    ssim = structural_similarity(X, X_estego, data_range=255)
    print(f"SSIM = {ssim:.4f}")
except ImportError:
    print("scikit-image não encontrado; rode 'pip install scikit-image' "
          "para também calcular o SSIM.")

# Plotagem

A implementação também exibe visualmente três imagens: a imagem original, a imagem esteganográfica e o mapa de diferenças entre ambas. Esse mapa evidencia que as alterações estão distribuídas de maneira muito sutil, confirmando que a informação foi inserida sem provocar distorções perceptíveis.

In [ ]:
plt.figure(figsize=(18,6))
plt.subplot(131)
plt.imshow(X, cmap='gray')
plt.title("Imagem Original")
plt.axis('off')
plt.subplot(132)
plt.imshow(X_estego, cmap='gray')
plt.title("Imagem Esteganográfica")
plt.axis('off')
plt.subplot(133)
plt.imshow(X_estego - X, cmap='gray')
plt.title("Diferença")
plt.colorbar()
plt.show()

# Aproximação por posto reduzido

Além disso, são geradas aproximações da imagem utilizando decomposição de posto reduzido, considerando diferentes quantidades de valores singulares (por exemplo, $r=100$ e $r=200$). Essas aproximações demonstram que grande parte da informação visual permanece preservada mesmo utilizando apenas uma parcela dos valores singulares, reforçando uma das principais propriedades da SVD: a concentração da energia da imagem nos maiores valores singulares.

In [ ]:
for r in (5,50,100, 200):
    # Reconstrói a imagem esteganográfica
    Xapprox = U[:, :r] @ np.diag(s_estego[:r]) @ VT[:r, :]

    plt.figure(figsize=(6, 3))

    # Imagem original
    plt.subplot(121)
    plt.imshow(X, cmap='gray')
    plt.title("Imagem Original")
    plt.axis('off')

    # Imagem esteganográfica
    plt.subplot(122)
    plt.imshow(Xapprox, cmap='gray')
    plt.title(f"Imagem Esteganográfica (r = {r})")
    plt.axis('off')

    plt.tight_layout()
    plt.show()

# Salvar Imagem Esteganografada

Por fim, a imagem esteganográfica é armazenada em um arquivo para que possa ser utilizada posteriormente no processo de decodificação da mensagem. Nessa etapa posterior, basta aplicar novamente a decomposição SVD à imagem modificada e verificar a paridade dos valores singulares escalonados para reconstruir a sequência de bits e, consequentemente, recuperar a mensagem original. Dessa forma, toda a implementação explora a estabilidade da SVD e a baixa sensibilidade visual a pequenas alterações nos valores singulares, permitindo ocultar informações de forma simples, eficiente e praticamente imperceptível.

In [ ]:
np.save("imagem_estego.npy", X_estego)
print("\nImagem salva como imagem_estego.png")

# Testes Exploratórios: Robustez e Esteganálise

As duas células a seguir **não fazem parte do fluxo principal** de codificação/decodificação; servem apenas para investigar, de forma preliminar, os objetivos de robustez e esteganálise declarados na Introdução do relatório.

In [ ]:
# Esteganálise simples: a distribuição de paridades dos valores
# singulares muda de forma perceptível após a inserção da mensagem?
def conta_paridade(valores, escala=100):
    pares = sum(1 for v in valores if int(round(v * escala)) % 2 == 0)
    return pares, len(valores) - pares

pares_antes, impares_antes = conta_paridade(s)
pares_depois, impares_depois = conta_paridade(s_estego)
print(f"Antes  - pares: {pares_antes}, ímpares: {impares_antes}")
print(f"Depois - pares: {pares_depois}, ímpares: {impares_depois}")

In [ ]:
# Teste de robustez: a mensagem sobrevive a uma recompressão JPEG
# da imagem esteganográfica? (limitação já discutida na comparação
# com Bergman e Davidson, 2005 — aqui ela é medida empiricamente)
from PIL import Image

Image.fromarray(np.clip(X_estego, 0, 255).astype(np.uint8)).save(
    "estego_jpeg_teste.jpg", quality=90
)
X_recompactada = np.array(
    Image.open("estego_jpeg_teste.jpg").convert("L"), dtype=np.float64
)

_, s_recompactada, _ = np.linalg.svd(X_recompactada, full_matrices=False)

bits_pos_jpeg = ""
for valor in reversed(s_recompactada):
    inteiro = int(round(valor * escala))
    bits_pos_jpeg += "0" if inteiro % 2 == 0 else "1"

mensagem_pos_jpeg = bits_para_texto(bits_pos_jpeg)
print("Mensagem após recompressão JPEG:", repr(mensagem_pos_jpeg))

In [ ]:
# Teste de quantização (rodar ANTES da célula de teste JPEG)
X_estego_uint8 = np.clip(np.round(X_estego), 0, 255).astype(np.uint8)

from PIL import Image
Image.fromarray(X_estego_uint8).save("estego_quantizado.png")

X_recarregada = np.array(Image.open("estego_quantizado.png"), dtype=np.float64)
_, s_quantizado, _ = np.linalg.svd(X_recarregada, full_matrices=False)

bits_pos_quant = ""
for valor in reversed(s_quantizado):
    inteiro = int(round(valor * escala))
    bits_pos_quant += "0" if inteiro % 2 == 0 else "1"

mensagem_pos_quant = bits_para_texto(bits_pos_quant)
print("Mensagem após quantização para uint8:", repr(mensagem_pos_quant))

L = len(bits_originais)
erros_quant = sum(b1 != b2 for b1, b2 in zip(bits_originais, bits_pos_quant[:L]))
print(f"Taxa de erro pós-quantização = {erros_quant/L:.4f} ({erros_quant}/{L} bits)")

# Decodificação

A etapa de decodificação tem como objetivo recuperar a mensagem secreta previamente inserida na imagem esteganográfica utilizando a **Decomposição em Valores Singulares (SVD)**. Como a codificação foi realizada alterando apenas a paridade dos valores singulares, a recuperação da mensagem consiste em decompor novamente a imagem e analisar esses valores.

# Funções auxiliares

Inicialmente, é definida a função `bits_para_texto()`, responsável por converter uma sequência de bits em uma cadeia de caracteres. A função percorre os bits em grupos de oito, correspondentes a um byte. Cada byte é convertido para um número inteiro em base dois e, em seguida, para seu respectivo caractere utilizando a função `chr()`. O processo continua até que seja encontrado o caractere nulo (`'\0'`), utilizado como marcador de término da mensagem, ou até que não existam mais bits suficientes para formar um byte completo.

In [ ]:
def bits_para_texto(bits):
    mensagem = ""
    for i in range(0, len(bits), 8):
        byte = bits[i:i+8]
        if len(byte) < 8:
            break
        caractere = chr(int(byte, 2))
        # fim da mensagem
        if caractere == '\0':
            break
        mensagem += caractere
    return mensagem

# Pré-Processsamento

Em seguida, a imagem esteganográfica é carregada a partir do arquivo `imagem_estego.npy`, que contém a matriz reconstruída durante o processo de codificação. Caso a imagem possua três canais de cor (RGB), ela é novamente convertida para tons de cinza através da média dos canais, garantindo que a matriz utilizada na extração seja compatível com aquela empregada durante a inserção da mensagem.

In [ ]:
X_estego = np.load("imagem_estego.npy") # Leitura da imagem esteganográfica

if len(X_estego.shape) == 3: # Caso seja RGB
    X_estego = np.mean(X_estego, axis=2)
plt.figure(figsize=(8, 4))
plt.imshow(X_estego, cmap="gray")
plt.title("Imagem Esteganográfica")
plt.axis("off")
plt.show()

# Calculo do SVD

Após o carregamento da imagem, aplica-se novamente a Decomposição em Valores Singulares,

$$
X_{\text{estego}} = U \Sigma_{\text{estego}} V^T,
$$

obtendo as matrizes ortogonais $U$ e $V^T$ e o vetor dos valores singulares contidos em $\Sigma_{\text{estego}}$. Como apenas esses valores foram modificados durante a codificação, basta analisá-los para recuperar a informação escondida.

In [ ]:
U, s, VT = np.linalg.svd(X_estego, full_matrices=False)

# Recuperação do bits

A recuperação da mensagem segue o processo inverso da codificação. Como a mensagem foi inserida a partir dos **menores** valores singulares (protegendo os maiores), a extração percorre o vetor de valores singulares recuperado **do menor para o maior** — ou seja, em ordem inversa à convenção padrão do NumPy. Isso preserva o caráter cego (*blind*) do método: o receptor não precisa saber o tamanho $L$ da mensagem original, pois a leitura sempre começa no menor valor singular e avança até encontrar o caractere nulo (`'\0'`), que sinaliza o fim da mensagem.

Cada valor singular é multiplicado pelo mesmo fator de escala utilizado na codificação (igual a 100) e arredondado para um número inteiro. Em seguida, verifica-se sua paridade:

- Se o número inteiro for **par**, o bit recuperado é **0**;
- Se o número inteiro for **ímpar**, o bit recuperado é **1**.

In [ ]:
escala = 100
bits_recuperados = ""
for valor in reversed(s):
    inteiro = int(round(valor * escala))
    if inteiro % 2 == 0:
        bits_recuperados += "0"
    else:
        bits_recuperados += "1"

# Conversão de bits para texto

Após recuperar todos os bits, a sequência binária é convertida novamente para texto utilizando a função `bits_para_texto()`. A conversão é realizada agrupando os bits em blocos de oito, formando bytes que representam os caracteres da mensagem em sua codificação ASCII. Quando o caractere nulo (`'\0'`) é encontrado, o algoritmo interrompe a leitura, indicando o final da mensagem.

Por fim, a mensagem recuperada é exibida ao usuário. Como o processo de codificação modificou apenas a paridade dos valores singulares e essas alterações foram suficientemente pequenas para não comprometer sua estabilidade numérica, a sequência de bits extraída coincide com aquela originalmente inserida, permitindo reconstruir integralmente o texto secreto.

Dessa forma, a implementação demonstra que a SVD pode ser utilizada tanto para ocultar quanto para recuperar informações em imagens digitais. A codificação altera apenas pequenas propriedades dos valores singulares, enquanto a decodificação explora essas mesmas propriedades para reconstruir a mensagem sem a necessidade de comparar a imagem esteganográfica com a imagem original, caracterizando um método de esteganografia cega (*blind steganography*).

In [ ]:
mensagem = bits_para_texto(bits_recuperados)

print("Mensagem recuperada:", mensagem)

# Avaliação: Taxa de Erro de Bits

Esta célula compara os bits originais com os bits recuperados para quantificar a taxa de erro de extração. **Ela só é possível aqui porque estamos na mesma sessão/kernel da codificação** — em um cenário real de esteganografia cega, o receptor não teria acesso a `bits_originais`; esta comparação serve apenas para fins de avaliação experimental do método.

In [ ]:
L = len(bits_originais)
comparaveis = bits_recuperados[:L]
erros = sum(b1 != b2 for b1, b2 in zip(bits_originais, comparaveis))
taxa_erro = erros / L
print(f"Taxa de erro de bits = {taxa_erro:.4f} ({erros}/{L} bits incorretos)")